<span style="color: #6a737d; font-family: monospace;">
Created on Tue Nov 12 2024 17:41:59<br>
Author: Mukai (Tom Notch) Yu<br>
Email: mukaiy@andrew.cmu.edu<br>
Affiliation: Carnegie Mellon University, Robotics Institute<br>
<br>
Copyright Ⓒ 2024 Mukai (Tom Notch) Yu<br>
</span>

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# %cd $USF_PROJECT_DIRECTORY doesn't work here because it's set by os.environ, not before the notebook starts
%cd ../..
%load_ext autoreload
%autoreload 2

import os.path as osp

import cv2
import healpy as hp
import numpy as np
from matplotlib import pyplot as plt

from usf.generate_lens_normal_map import generate_equirectangular_normal_map
from usf.sampler.sampler import SphericalSampler
from usf.sampler.value.value_sampler import ValueSampler
from usf.utils.files import read_file
from usf.utils.spherical import average_pixel_area
from usf.utils.spherical_image import BatchSphericalImage, SphericalImage
from usf.visualization.spherical_projection import visualize_spherical_image

## Import A Photo

In [ ]:
CURRENT_DIR = osp.dirname(osp.realpath("__file__"))
CAMERA_CFG_PATH = osp.join(CURRENT_DIR, "config/wildfire/subcanopy/subcanopy.yaml")
INPUT_IMAGE_PATH = osp.join(
    CURRENT_DIR, "config/wildfire/subcanopy/sensors/camera_0_sample.png"
)

In [ ]:
# Load the camera configuration
camera_config = read_file(CAMERA_CFG_PATH)["sensor_configs"]["rgb_0"]
print("Camera Config:")
print(camera_config)

# lens normal map
lens_normal_map = camera_config["lens_normal_map"]

# mask
mask = camera_config["mask"]

In [ ]:
# Load example image
input_image = read_file(INPUT_IMAGE_PATH)
input_image = cv2.resize(
    input_image, (camera_config["image_width"], camera_config["image_height"])
)

# Display one of the images
plt.axis("off")
plt.imshow(
    input_image,
)

In [ ]:
mask_gray = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
_, binary_mask = cv2.threshold(mask_gray, 128, 255, cv2.THRESH_BINARY)
binary_mask = binary_mask.astype(bool)

In [ ]:
image_color = input_image[binary_mask].reshape(-1, 3)
image_vector = lens_normal_map[binary_mask].reshape(-1, 3)
spherical_image = SphericalImage(value=image_color, vector=image_vector)

In [ ]:
average_pixel_area(spherical_image.vector)

In [ ]:
visualize_spherical_image(spherical_image, point_size=5)

## Fibonacci Spiral Discretization

In [ ]:
fibonacci_sampler = SphericalSampler(
    {
        "location_sampler": "fibonacci",
        "value_sampler": "nearest_neighbor",
    }
)

In [ ]:
fibonacci_sample_spherical_image = fibonacci_sampler(spherical_image)

In [ ]:
visualize_spherical_image(
    fibonacci_sample_spherical_image,
    point_size=5,
)

## Quasi-Random Discretization

In [ ]:
quasirandom_sampler = SphericalSampler(
    {
        "location_sampler": "quasirandom",
        "value_sampler": "nearest_neighbor",
    }
)

In [ ]:
quasirandom_sample_spherical_image = quasirandom_sampler(spherical_image)

In [ ]:
visualize_spherical_image(
    quasirandom_sample_spherical_image,
    point_size=5,
)

## HEALPix Discretization

In [ ]:
NSIDE = 128  # Resolution parameter; higher values mean higher resolution
NPIX = hp.nside2npix(NSIDE)  # Total number of pixels
m = np.arange(NPIX)  # Example map with pixel values equal to their indices

In [ ]:
hp.mollview(m, title="Mollweide Projection")
plt.show()

In [ ]:
healpix_sampler = SphericalSampler(
    {
        "location_sampler": "healpix",
        "value_sampler": "nearest_neighbor",
    }
)

In [ ]:
healpix_sample_spherical_image = healpix_sampler(spherical_image)

In [ ]:
visualize_spherical_image(
    healpix_sample_spherical_image,
    point_size=5,
)

## Tetrahedron Discretization

In [ ]:
tetrahedron_sampler = SphericalSampler(
    {
        "location_sampler": "tetrahedron",
        "value_sampler": "nearest_neighbor",
    }
)

In [ ]:
tetrahedron_sample_spherical_image = tetrahedron_sampler(spherical_image)

In [ ]:
visualize_spherical_image(tetrahedron_sample_spherical_image, point_size=5)

## Hexahedron Discretization

In [ ]:
hexahedron_sampler = SphericalSampler(
    {
        "location_sampler": "hexahedron",
        "value_sampler": "nearest_neighbor",
    }
)

In [ ]:
hexahedron_sample_spherical_image = hexahedron_sampler(spherical_image)

In [ ]:
visualize_spherical_image(hexahedron_sample_spherical_image, point_size=5)

## Octahedron Discretization

In [ ]:
octahedron_sampler = SphericalSampler(
    {
        "location_sampler": "octahedron",
        "value_sampler": "nearest_neighbor",
    }
)

In [ ]:
octahedron_sample_spherical_image = octahedron_sampler(spherical_image)

In [ ]:
visualize_spherical_image(octahedron_sample_spherical_image, point_size=5)

## Icosahedron Discretization

In [ ]:
icosahedron_sampler = SphericalSampler(
    {
        "location_sampler": "icosahedron",
        "value_sampler": "nearest_neighbor",
        # "reject_oo_fov_vector": False,
        # "reject_oo_fov_value": True,
        # "output_average_pixel_area": 6.817692440406791e-06,  # matching density of PANDORA dataset
    }
)

In [ ]:
icosahedron_sample_spherical_image = icosahedron_sampler(spherical_image)

In [ ]:
visualize_spherical_image(icosahedron_sample_spherical_image)

## Equirectangular Discretization

In [ ]:
equirectangular_sampler = SphericalSampler(
    {
        "location_sampler": "equirectangular",
        "value_sampler": "nearest_neighbor",
    }
)

In [ ]:
equirectangular_sample_spherical_image = equirectangular_sampler(spherical_image)

In [ ]:
visualize_spherical_image(equirectangular_sample_spherical_image, point_size=5)

## Override Output Location

Sample a PANDORA-like image from a random spherical image

In [ ]:
value_sampler = ValueSampler(
    {"value_sampler": "nearest_neighbor", "reject_oo_fov_value": True}
)

In [ ]:
override_sample_spherical_image = value_sampler(
    BatchSphericalImage(spherical_image),
    generate_equirectangular_normal_map(height=960, width=1920),
)

In [ ]:
visualize_spherical_image(override_sample_spherical_image.apply_mask())